# Graphify and label ESM embeddings for PST

For CheckAMG PST model training, we need to:

* Convert ESM embeddings to PST-formatted embeddings (using the ESM embeddings generated in the notebook `esm_embed_data.ipynb`)
* Add labels to the PST embeddings (`strand`, `ptr`, `sizes`, `protein_is_viral`, `protein_is_AVG`, `scaffold_category`)
* Calculate average protein embeddings over each genome/scaffold to mark proteins for the validation set

Nothing else needs to change in the test dataset embeddings.

In [1]:
! pip install polars pyfastatools uv faiss-cpu igraph tqdm --quiet
! uv pip install torch --quiet
! uv pip install ptn-set-transformer --no-build-isolation --quiet

## Add labels to protein embeddings

In [2]:
from __future__ import annotations

import re
from pathlib import Path
from collections import defaultdict

import numpy as np
import polars as pl
import tables as tb
import torch
from tqdm import tqdm
from pyfastatools import Parser
from torch_geometric.utils import segment

In [3]:
from pathlib import Path
ROOT_DIR = Path("/storage2/scratch/kosmopoulos/projects/checkAMG/training_data")
SPLIT_DIR = ROOT_DIR.joinpath("train_test_splits")
PST_OUT = ROOT_DIR.parent.joinpath("pst/training")
TRAIN_OUT = PST_OUT.joinpath("train_data")
TEST_OUT = PST_OUT.joinpath("test_data")
ESM_DIR = PST_OUT.joinpath("esm")
TRAIN_OUT = PST_OUT.joinpath("train_data")
TEST_OUT = PST_OUT.joinpath("test_data")
TRAIN_FASTA = TRAIN_OUT.joinpath("train_ptns.faa")
TRAIN_SPLIT_DIR = TRAIN_OUT.joinpath("train_split")

In [4]:
MAX_CONTIG_SIZE = 2048

In [5]:
def fasta_has_fragments(faa: Path) -> bool:
    for h in Parser(str(faa)).headers():
        if "_FRAGMENT_" in h.name:
            return True
    return False

def get_fragment_ptr(faa: Path) -> torch.Tensor:
    num_fragments: dict[str, int] = defaultdict(int)
    for h in Parser(str(faa)).headers():
        base = h.name.split("_FRAGMENT_")[0]
        num_fragments[base] += 1

    sizes = np.fromiter(num_fragments.values(), dtype=np.int64)
    ptr = np.zeros(len(sizes) + 1, dtype=np.int64)
    ptr[1:] = np.cumsum(sizes, dtype=np.int64)
    return torch.from_numpy(ptr)

In [6]:
def find_train_embed_files(esm_dir: Path) -> list[Path]:
    dirs = [p for p in esm_dir.iterdir() if p.is_dir() and re.match(r"^esm_train_data_\d+$", p.name)]
    dirs = sorted(dirs, key=lambda p: int(p.name.split("_")[-1]))

    out = []
    for d in dirs:
        h5s = list(d.glob("*.h5"))
        if len(h5s) != 1:
            raise RuntimeError(f"[train] expected exactly 1 h5 in {d}, found {len(h5s)}")
        out.append(h5s[0])

    if not out:
        raise RuntimeError(f"No train embedding dirs found under {esm_dir}")
    return out

def find_test_embed_file(esm_dir: Path, name: str) -> Path:
    d = esm_dir / f"esm_test_data_{name}"
    if not d.exists() or not d.is_dir():
        raise RuntimeError(f"[{name}] missing test embedding dir: {d}")

    h5s = list(d.glob("*.h5"))
    if len(h5s) != 1:
        raise RuntimeError(f"[{name}] expected exactly 1 h5 in {d}, found {len(h5s)}")
    return h5s[0]

In [7]:
def train_chunk_fastas(train_split_dir: Path) -> list[Path]:
    chunks = list(train_split_dir.glob("train_chunk_*.faa"))
    if not chunks:
        raise RuntimeError(f"No train_chunk_*.faa found in {train_split_dir}")
    chunks = sorted(chunks, key=lambda p: int(p.stem.split("_")[-1]))
    return chunks

In [8]:
train_chunk_faas = train_chunk_fastas(TRAIN_SPLIT_DIR)
train_embed_h5s = find_train_embed_files(ESM_DIR)

print("train chunks:", len(train_chunk_faas))
print("train embed h5s:", len(train_embed_h5s))
assert len(train_chunk_faas) == len(train_embed_h5s)

train chunks: 15
train embed h5s: 15


In [9]:
def compute_scaffold_ptr_from_fasta(faa: Path) -> tuple[np.ndarray, np.ndarray]:
    # scaffold = protein header name without final "_NUMBER"
    scaffold_sizes_map: dict[str, int] = defaultdict(int)
    for h in Parser(str(faa)).headers():
        scaffold = h.name.rsplit("_", 1)[0]
        scaffold_sizes_map[scaffold] += 1

    sizes = np.fromiter(scaffold_sizes_map.values(), dtype=np.int64)
    ptr = np.zeros(len(sizes) + 1, dtype=np.int64)
    ptr[1:] = np.cumsum(sizes, dtype=np.int64)
    return sizes, ptr

def build_metadata_from_split_df(df: pl.DataFrame) -> pl.DataFrame:
    # Ensure correct order: must match FASTA order
    # Sort by Contig/gene_number
    df = df.sort(["Contig", "gene_number"])

    # pid: row index in this sorted order
    # gid: contig id in this order
    meta = (
        df.select(["Contig", "Protein", "frame", "viral", "AVG"])
        .with_row_index("pid")
        .with_columns(
            gid=pl.col("Contig").rle_id(),
            strand=pl.col("frame").cast(pl.Int32),
            protein_is_viral=pl.col("viral").cast(pl.Boolean),
            protein_is_AVG=pl.col("AVG").cast(pl.Boolean),
        )
        .select(["pid", "gid", "Contig", "Protein", "strand", "protein_is_viral", "protein_is_AVG"])
    )

    contig_flags = meta.group_by("gid").agg([
        (~pl.col("protein_is_viral")).any().alias("has_nonviral"),
        pl.col("protein_is_AVG").any().alias("has_AVG"),
    ])

    meta = (
        meta.join(contig_flags, on="gid", how="left")
        .select(["pid", "gid", "strand", "protein_is_viral", "protein_is_AVG", "has_nonviral", "has_AVG"])
        .sort("pid")
    )
    return meta

def write_graph_arrays(fp: tb.File, arrays: dict[str, tuple[np.ndarray, tb.Atom]]):
    # overwrite if present, then write
    for name, (arr, atom) in arrays.items():
        if hasattr(fp.root, name):
            fp.remove_node(fp.root, name)
        fp.create_carray(fp.root, name, atom=atom, obj=arr)

def write_graphfmt_h5(
    *,
    name: str,
    out_file: Path,
    fasta_files: list[Path],
    embed_files: list[Path],
    final_fasta: Path,
    split_df: pl.DataFrame,
) -> None:
    assert len(fasta_files) == len(embed_files)

    needs_index = {faa: fasta_has_fragments(faa) for faa in fasta_files}
    total_expected = int(split_df.height)

    print(f"[{name}] out_file: {out_file}")
    print(f"[{name}] n_proteins (df): {total_expected:,}")
    print(f"[{name}] n_embed_files: {len(embed_files)}")
    print(f"[{name}] final_fasta: {final_fasta}")
    print("")

    # figure embedding dim from first file
    with tb.open_file(str(embed_files[0])) as f0:
        emb_dim = int(f0.root.data.shape[1])

    out_file.parent.mkdir(parents=True, exist_ok=True)

    # 1) write /data (append embeddings in correct order)
    try:
        tb.file._open_files.close_all()
    except Exception:
        pass

    with tb.open_file(str(out_file), "w") as fdst:
        storage = fdst.create_earray(
            where=fdst.root,
            name="data",
            atom=tb.Float32Atom(),
            expectedrows=total_expected,
            shape=(0, emb_dim),
            filters=tb.Filters(complevel=5, complib="blosc2:lz4hc"),
        )

        for ef, faa in tqdm(list(zip(embed_files, fasta_files)), desc=f"[{name}] append embeddings"):
            with tb.open_file(str(ef)) as fsrc:
                emb = torch.from_numpy(fsrc.root.data[:])

            if needs_index[faa]:
                ptr = get_fragment_ptr(faa)
                emb = segment(emb, ptr, reduce="mean")

            storage.append(emb.numpy().astype(np.float32))

        n_rows = int(fdst.root.data.shape[0])
        if n_rows != total_expected:
            raise RuntimeError(f"[{name}] wrote {n_rows:,} rows but expected {total_expected:,} (order/count bug)")

    # 2) compute scaffold ptr/sizes from final fasta (should match df order)
    scaffold_sizes, scaffold_ptr = compute_scaffold_ptr_from_fasta(final_fasta)

    # 3) build metadata arrays from split_df
    meta = build_metadata_from_split_df(split_df)

    strand = meta["strand"].to_numpy().astype(np.int64)
    protein_is_viral = meta["protein_is_viral"].to_numpy().astype(np.bool_)
    protein_is_AVG = meta["protein_is_AVG"].to_numpy().astype(np.bool_)

    # validate scaffold partitioning against n_rows
    if int(scaffold_sizes.sum()) != total_expected:
        raise RuntimeError(f"[{name}] scaffold_sizes.sum()={int(scaffold_sizes.sum()):,} != expected={total_expected:,}")

    # scaffold flags (fast, from meta summary)
    scaffold_summary = (
        meta.select(["gid", "has_nonviral", "has_AVG"])
        .unique()
        .sort("gid")
    )
    scaffold_has_nonviral = scaffold_summary["has_nonviral"].to_numpy().astype(np.bool_)
    scaffold_has_AVG = scaffold_summary["has_AVG"].to_numpy().astype(np.bool_)

    if scaffold_sizes.shape[0] != scaffold_has_AVG.shape[0]:
        raise RuntimeError(
            f"[{name}] n_scaffolds mismatch: sizes={scaffold_sizes.shape[0]} vs flags={scaffold_has_AVG.shape[0]}"
        )

    # 4) write arrays into the same h5
    try:
        tb.file._open_files.close_all()
    except Exception:
        pass

    with tb.open_file(str(out_file), "a", filters=tb.Filters(complevel=5, complib="blosc2:lz4hc")) as fp:
        n_rows = int(fp.root.data.shape[0])
        if int(scaffold_sizes.sum()) != n_rows:
            raise RuntimeError(f"[{name}] sum(sizes) != n_rows ({int(scaffold_sizes.sum())} vs {n_rows})")

        arrays = {
            "sizes": (np.asarray(scaffold_sizes, dtype=np.int64), tb.Int64Atom()),
            "ptr": (np.asarray(scaffold_ptr, dtype=np.int64), tb.Int64Atom()),
            "strand": (np.asarray(strand, dtype=np.int64), tb.Int64Atom()),
            "protein_is_viral": (np.asarray(protein_is_viral, dtype=np.bool_), tb.BoolAtom()),
            "protein_is_AVG": (np.asarray(protein_is_AVG, dtype=np.bool_), tb.BoolAtom()),
            "scaffold_has_nonviral": (np.asarray(scaffold_has_nonviral, dtype=np.bool_), tb.BoolAtom()),
            "scaffold_has_AVG": (np.asarray(scaffold_has_AVG, dtype=np.bool_), tb.BoolAtom()),
        }
        write_graph_arrays(fp, arrays)

    print(f"[{name}] done: wrote graphfmt h5 with {total_expected:,} proteins\n")

In [10]:
import glob
import os

parquet_paths = glob.glob(os.path.join(TRAIN_OUT, "*.parquet"))
parquet_paths += glob.glob(os.path.join(TEST_OUT, "*.parquet"))

# sort so "train" is first, everything else alphabetical after
parquet_paths = sorted(
    parquet_paths,
    key=lambda p: (os.path.basename(p).replace(".parquet", "") != "train",
                   os.path.basename(p))
)

final_dfs = {}

for dataset_path in parquet_paths:
    dataset_name = os.path.basename(dataset_path).replace(".parquet", "").replace("_ptns", "")
    print(f"Loading {dataset_name} from {dataset_path}")
    final_dfs[dataset_name] = (pl.read_parquet(dataset_path))

Loading test_equal_pos from /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_pos_ptns.parquet
Loading test_equal_source from /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_source_ptns.parquet
Loading test_half_virus_host from /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_half_virus_host_ptns.parquet
Loading test_host_enriched from /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_host_enriched_ptns.parquet
Loading test_input from /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_input_ptns.parquet
Loading test_mge_enriched from /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_mge_enriched_ptns.parquet
Loading test_near_all_host from /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_near_all_host_ptns.parquet
Loading test_near_all_virus from /storage2/scratch/kosmopoulos/projects/checkAM

In [11]:
TRAIN_H5 = TRAIN_OUT.joinpath("checkAMG_train_esm2_t30_150M.graphfmt.h5")

write_graphfmt_h5(
    name="train",
    out_file=TRAIN_H5,
    fasta_files=train_chunk_faas,
    embed_files=train_embed_h5s,
    final_fasta=TRAIN_FASTA,
    split_df=final_dfs["train"],
)

[train] out_file: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/checkAMG_train_esm2_t30_150M.graphfmt.h5
[train] n_proteins (df): 13,942,367
[train] n_embed_files: 15
[train] final_fasta: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/train_data/train_ptns.faa



[train] append embeddings: 100%|██████████| 15/15 [16:40<00:00, 66.70s/it]


[train] done: wrote graphfmt h5 with 13,942,367 proteins



In [12]:
test_names = [k for k in final_dfs.keys() if k != "train"]

for name in test_names:
    test_faa = TEST_OUT.joinpath(f"{name}_ptns.faa")
    if not test_faa.exists():
        raise FileNotFoundError(f"Missing test FASTA: {test_faa}")

    test_embed = find_test_embed_file(ESM_DIR, name)
    TEST_H5 = TEST_OUT.joinpath(f"checkAMG_{name}_esm2_t30_150M.graphfmt.h5")

    write_graphfmt_h5(
        name=name,
        out_file=TEST_H5,
        fasta_files=[test_faa],
        embed_files=[test_embed],
        final_fasta=test_faa,
        split_df=final_dfs[name],
    )

[test_equal_pos] out_file: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/checkAMG_test_equal_pos_esm2_t30_150M.graphfmt.h5
[test_equal_pos] n_proteins (df): 380,925
[test_equal_pos] n_embed_files: 1
[test_equal_pos] final_fasta: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_pos_ptns.faa



[test_equal_pos] append embeddings: 100%|██████████| 1/1 [00:25<00:00, 25.79s/it]


[test_equal_pos] done: wrote graphfmt h5 with 380,925 proteins

[test_equal_source] out_file: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/checkAMG_test_equal_source_esm2_t30_150M.graphfmt.h5
[test_equal_source] n_proteins (df): 190,143
[test_equal_source] n_embed_files: 1
[test_equal_source] final_fasta: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_equal_source_ptns.faa



[test_equal_source] append embeddings: 100%|██████████| 1/1 [00:13<00:00, 13.48s/it]


[test_equal_source] done: wrote graphfmt h5 with 190,143 proteins

[test_half_virus_host] out_file: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/checkAMG_test_half_virus_host_esm2_t30_150M.graphfmt.h5
[test_half_virus_host] n_proteins (df): 632,631
[test_half_virus_host] n_embed_files: 1
[test_half_virus_host] final_fasta: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_half_virus_host_ptns.faa



...

[test_near_all_virus] done: wrote graphfmt h5 with 841,115 proteins

[test_provirus] out_file: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/checkAMG_test_provirus_esm2_t30_150M.graphfmt.h5
[test_provirus] n_proteins (df): 75,260
[test_provirus] n_embed_files: 1
[test_provirus] final_fasta: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_provirus_ptns.faa



[test_provirus] append embeddings: 100%|██████████| 1/1 [00:04<00:00,  4.86s/it]


[test_provirus] done: wrote graphfmt h5 with 75,260 proteins

[test_virus_enriched] out_file: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/checkAMG_test_virus_enriched_esm2_t30_150M.graphfmt.h5
[test_virus_enriched] n_proteins (df): 425,458
[test_virus_enriched] n_embed_files: 1
[test_virus_enriched] final_fasta: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/test_virus_enriched_ptns.faa



[test_virus_enriched] append embeddings: 100%|██████████| 1/1 [00:29<00:00, 29.06s/it]


[test_virus_enriched] done: wrote graphfmt h5 with 425,458 proteins



## Verify each output H5 is readable by PST

In [13]:
import pst

def test_pst_readable(h5_path: Path, label: str) -> None:
    ds = pst.LazyGenomeDataset(file=str(h5_path))
    # minimal touches that will fail fast if metadata is wrong
    n_nodes = int(ds.data.shape[0])
    n_scaff = int(ds.sizes.shape[0])
    print(f"[{label}] nodes={n_nodes:,} scaffolds={n_scaff:,} emb_dim={int(ds.data.shape[1])}")

test_pst_readable(TRAIN_H5, "train")
for name in test_names:
    test_pst_readable(TEST_OUT.joinpath(f"checkAMG_{name}_esm2_t30_150M.graphfmt.h5"), name)


[train] nodes=13,942,367 scaffolds=1,139,814 emb_dim=640
[test_equal_pos] nodes=380,925 scaffolds=42,092 emb_dim=640
[test_equal_source] nodes=190,143 scaffolds=20,738 emb_dim=640
[test_half_virus_host] nodes=632,631 scaffolds=68,229 emb_dim=640
[test_host_enriched] nodes=311,522 scaffolds=35,183 emb_dim=640
[test_input] nodes=953,918 scaffolds=108,568 emb_dim=640
[test_mge_enriched] nodes=203,361 scaffolds=19,441 emb_dim=640
[test_near_all_host] nodes=665,307 scaffolds=88,377 emb_dim=640
[test_near_all_virus] nodes=841,115 scaffolds=78,563 emb_dim=640
[test_provirus] nodes=75,260 scaffolds=3,069 emb_dim=640
[test_virus_enriched] nodes=425,458 scaffolds=42,826 emb_dim=640


/tmp/ipykernel_2463214/4042306934.py:6: DeprecationWarning: Deprecated since v1.3: Use .protein_data instead for clarity
  n_nodes = int(ds.data.shape[0])
/tmp/ipykernel_2463214/4042306934.py:7: DeprecationWarning: Deprecated since v1.3: Use .scaffold_sizes instead for clarity
  n_scaff = int(ds.sizes.shape[0])
/tmp/ipykernel_2463214/4042306934.py:8: DeprecationWarning: Deprecated since v1.3: Use .protein_data instead for clarity
  print(f"[{label}] nodes={n_nodes:,} scaffolds={n_scaff:,} emb_dim={int(ds.data.shape[1])}")


## Compute the genome (scaffold) average embeddings

### Load protein embeddings

In [14]:
import tables as tb
import torch
from torch_geometric.utils import segment
import numpy as np
import igraph as ig
import random
import faiss
from numpy.typing import NDArray

FloatArray = NDArray[np.float32]
BoolArray = NDArray[np.bool_]

faiss.omp_set_num_threads(128)

In [15]:
try:
    tb.file._open_files.close_all()
except Exception:
    pass

with tb.open_file(TRAIN_H5) as fp:
    protein_embeddings = torch.from_numpy(fp.root.data[:])
    scaffold_ptr = torch.from_numpy(fp.root.ptr[:])

protein_embeddings.shape, scaffold_ptr.shape

/storage2/scratch/kosmopoulos/miniconda3/envs/pst_notebooks/lib/python3.11/site-packages/tables/file.py:130: UnclosedFileWarning: Closing remaining open file: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/checkAMG_test_near_all_host_esm2_t30_150M.graphfmt.h5
  warnings.warn(UnclosedFileWarning(msg))
/storage2/scratch/kosmopoulos/miniconda3/envs/pst_notebooks/lib/python3.11/site-packages/tables/file.py:130: UnclosedFileWarning: Closing remaining open file: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/checkAMG_test_provirus_esm2_t30_150M.graphfmt.h5
  warnings.warn(UnclosedFileWarning(msg))
/storage2/scratch/kosmopoulos/miniconda3/envs/pst_notebooks/lib/python3.11/site-packages/tables/file.py:130: UnclosedFileWarning: Closing remaining open file: /storage2/scratch/kosmopoulos/projects/checkAMG/pst/training/test_data/checkAMG_test_equal_source_esm2_t30_150M.graphfmt.h5
  warnings.warn(UnclosedFileWarning(msg))
/storage2/scratch/kosmopoulo

(torch.Size([13942367, 640]), torch.Size([1139815]))

### Average protein embeddings over each genome (scaffold)

In [16]:
genome_embedding = segment(protein_embeddings, scaffold_ptr, reduce="mean")
genome_embedding.shape

torch.Size([1139814, 640])

### Cluster genome embeddings

In [17]:
n_genomes = genome_embedding.shape[0]
n_genomes

1139814

In [18]:
norm_genome_embedding = torch.nn.functional.normalize(genome_embedding, dim=-1, p=2)

In [19]:
dim = norm_genome_embedding.shape[-1]
quantizer = faiss.IndexFlatIP(dim)
n_cells = 5000
index = faiss.IndexIVFFlat(quantizer, dim, n_cells, faiss.METRIC_INNER_PRODUCT)
index.train(norm_genome_embedding)
index.add(norm_genome_embedding)

### Find the 15 nearest neighbors based on cosine similarity

In [20]:
csim, knn = index.search(norm_genome_embedding, k=16)
csim

array([[0.99999994, 0.9985475 , 0.9984581 , ..., 0.99796146, 0.99795365,
        0.9979363 ],
       [0.99999994, 0.99479175, 0.99402195, ..., 0.99177206, 0.9917629 ,
        0.9917607 ],
       [1.        , 0.99935997, 0.99863744, ..., 0.9979949 , 0.997929  ,
        0.997898  ],
       ...,
       [1.        , 0.9973299 , 0.99697155, ..., 0.9965149 , 0.99641013,
        0.99639297],
       [1.0000002 , 0.9980964 , 0.99799615, ..., 0.9973553 , 0.9973211 ,
        0.997293  ],
       [0.9999998 , 0.995323  , 0.99491715, ..., 0.9940185 , 0.99387586,
        0.99387115]], shape=(1139814, 16), dtype=float32)

### Compute the angular similarity from the cosine similarity

In [21]:
csim: FloatArray = np.clip(csim, -1.0, 1.0)
asim = 1.0 - (np.arccos(csim) / np.pi)
asim

array([[0.9998901 , 0.9828416 , 0.98232126, ..., 0.97967184, 0.9796329 ,
        0.9795468 ],
       [0.9998901 , 0.9674988 , 0.96517736, ..., 0.959139  , 0.95911616,
        0.9591107 ],
       [1.        , 0.98861086, 0.98338145, ..., 0.97983927, 0.9795104 ,
        0.97935766],
       ...,
       [1.        , 0.97673374, 0.9752209 , ..., 0.9734174 , 0.9730205 ,
        0.972956  ],
       [1.        , 0.9803564 , 0.9798456 , ..., 0.97684467, 0.9766956 ,
        0.97657347],
       [0.9998096 , 0.96920234, 0.96789277, ..., 0.9651673 , 0.96475405,
        0.96474046]], shape=(1139814, 16), dtype=float32)

## Construct an edge weighted knn graph from the results and cluster

In [22]:
n, k = asim.shape
row = np.arange(n).repeat(k)
col = knn.flatten()
mask = col == -1
row = row[~mask]
col = col[~mask]

sim = asim.flatten()[~mask]

graph: ig.Graph = ig.Graph(
    directed=False,
    n=n,
    edges=zip(row, col),
    edge_attrs={"weight": sim},
)

graph.summary()

'IGRAPH U-W- 1139814 18235321 -- \n+ attr: weight (e)'

In [23]:
SEED = 20260220
random.seed(SEED)
ig.set_random_number_generator(random)

clusters = graph.community_leiden(
    weights="weight",
    resolution=0.75,
)

In [24]:
membership = np.array(clusters.membership)
membership.shape

(1139814,)

In [25]:
vc = clusters.as_cover()

Number of clusters:

In [26]:
len(vc)

395343

Inspect assigned to clusters 0, 1, 2:

In [27]:
vc[0], vc[1], vc[2]

([0, 8029, 753223],
 [1, 444950, 925542],
 [2, 2653, 4087, 4525, 4860, 4866, 10198, 15345, 16371, 16374, 178852, 449173])

Number of singletons:

In [28]:
num_singletons = sum(len(c) == 1 for c in vc)
num_singletons

123879

In [29]:
num_singletons / genome_embedding.shape[0]

0.1086835220483342

The % of singletons is conveniently close to a good % for splitting into train/val, so they will be used for the validation set.

## Assign singletons to the validation set in the training data

In [30]:
val_mask = np.ones(genome_embedding.shape[0], dtype=bool)
val_mask

array([ True,  True,  True, ...,  True,  True,  True], shape=(1139814,))

In [31]:
for cluster in vc:
    if len(cluster) == 1:
        continue

    val_mask[np.array(cluster)] = False

val_mask

array([False, False, False, ..., False, False, False], shape=(1139814,))

Should currently equal the number of singleton clusters:

In [32]:
assert val_mask.sum() == num_singletons

These may need to be added from the table:

In [33]:
with tb.open_file(TRAIN_H5) as fp:
    scaffold_has_nonviral: BoolArray = fp.root.scaffold_has_nonviral[:]
    scaffold_has_avg: BoolArray = fp.root.scaffold_has_AVG[:]

scaffold_has_nonviral.shape, scaffold_has_avg.shape # same as n_genomes

((1139814,), (1139814,))

Proportion of scaffolds with nonviral seqs and with AVGs:

In [34]:
scaffold_has_nonviral.sum() / n_genomes, scaffold_has_avg.sum() / n_genomes

(np.float64(0.5740822625445906), np.float64(0.36292061687257743))

Proportions of CURRENT val set (i.e. the singletons) that have non-viral or an AVG (want the nonviral prop to be close to the nonviral prop in the training set):

In [35]:
scaffold_has_nonviral[val_mask].sum() / val_mask.sum(), scaffold_has_avg[val_mask].sum() / val_mask.sum()

(np.float64(0.6570605187319885), np.float64(0.3688922254780875))

Close enough.

## Overwrite the PST embeddings with the version that has the validation mask

In [36]:
with tb.open_file(TRAIN_H5, "a") as fp:
    # need to remove it just in case it was there from other precomputed val masks, ie testing params
    if "scaffold_val_mask" in fp.root:
        fp.remove_node("/", "scaffold_val_mask")
        
    fp.create_carray(
        fp.root, 
        "scaffold_val_mask",
        obj=val_mask,
        filters=tb.Filters(complib="blosc2:lz4hc", complevel=5),
    )

In [37]:
with tb.open_file(TRAIN_H5) as fp:
    assert "scaffold_val_mask" in fp.root

Now, both the train (`checkAMG_train_esm2_t30_150M.graphfmt.h5`) and test (`checkAMG_test_<TEST DATASET>_esm2_t30_150M.graphfmt.h5`) protein embeddings at should be ready to use for training with PST.